# 🚀 Fine-Tuning Small Language Models with LoRA / QLoRA
### End-to-End Training, Baseline Evaluation, and Serving Pipeline

This notebook trains a 1B–3B parameter open-weight Small Language Model (e.g. `Qwen2.5-1.5B-Instruct` or `Llama-3.2-1B-Instruct`) using **4-bit QLoRA** (Quantized Low-Rank Adaptation) on a single free GPU (Google Colab T4 or Kaggle P100).

## 1. Environment & GPU Verification

In [ ]:
!nvidia-smi
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate evaluate scikit-learn pyyaml sentence-transformers faiss-cpu gradio

## 2. Clone Repository or Mount Drive

In [ ]:
import os
# If using Google Colab with Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

# Verify PyTorch and CUDA
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 3. Data Preparation & Splitting (80/10/10)

In [ ]:
import json
from pathlib import Path

# Load sample or target dataset
sample_path = Path("data/sample/domain_qa_sample.jsonl")
if not sample_path.exists():
    print("Creating sample data directory...")
    sample_path.parent.mkdir(parents=True, exist_ok=True)

print("Inspecting dataset schema...")
with open(sample_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 2:
            print(json.loads(line))
        else:
            break

## 4. Run Sanity Check Overfit Test (Step 7)

In [ ]:
# Overfit on 50 samples to confirm gradients propagate and loss descends
!python -m src.training.train_qlora --config configs/sanity_check.yaml

## 5. Main QLoRA Training Run

In [ ]:
# Run baseline QLoRA training
!python -m src.training.train_qlora --config configs/base_config.yaml

## 6. Evaluation & Benchmarking

In [ ]:
from src.evaluation.evaluator import ModelEvaluator
from src.data.dataset_loader import load_jsonl_dataset

test_ds = load_jsonl_dataset("data/sample/domain_qa_sample.jsonl")
records = [item for item in test_ds]

# Evaluate Base Model (Zero-Shot)
evaluator_base = ModelEvaluator(base_model_id="Qwen/Qwen2.5-0.5B-Instruct")
base_summary = evaluator_base.evaluate_test_set(records, output_report_path="results/metrics/base_model_eval.json")
print("Base Model Summary:", base_summary)

# Evaluate LoRA Tuned Model
# evaluator_lora = ModelEvaluator(base_model_id="Qwen/Qwen2.5-0.5B-Instruct", adapter_path="outputs/sanity_check/adapter")
# lora_summary = evaluator_lora.evaluate_test_set(records, output_report_path="results/metrics/qlora_eval.json")
# print("LoRA Tuned Summary:", lora_summary)

## 7. Merge Adapter & Launch Demo

In [ ]:
# Launch Gradio interactive demo with public shareable link
# !python src/serving/app.py